# US Bank Data: compare an exact quarter

Use three real, pinned sample institutions to demonstrate a same-quarter comparison and SQL. They have very different sizes and business models: **this is not a matched peer group, ranking or investment recommendation**. Frozen report date: **2026-06-30**.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("usbankdata") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "https://github.com/anshuman-pandey/us-bank-data/releases/download/v0.1.1/us_bank_data-0.1.1-py3-none-any.whl"])


In [ ]:
from usbankdata import load_sample

panels = [load_sample(cert) for cert in (628, 3510, 58665)]
rows = [panel.rows[-1] for panel in panels]
assert {row["report_date"] for row in rows} == {"2026-06-30"}
for row in rows:
    print(row["cert"], row["bank_name"], row["report_date"], row["assets_usd"])


## Query the table using standard-library SQL

Use an explicit CERT, not a ticker or similarly named parent. Leave missing ratios as SQL NULL. Zero and negative observations remain valid.


In [ ]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE banks (cert INTEGER, bank_name TEXT, report_date TEXT, assets_usd INTEGER, deposits_usd INTEGER, cet1_ratio_pct REAL, leverage_ratio_pct REAL, roa_pct REAL)")
columns = ["cert", "bank_name", "report_date", "assets_usd", "deposits_usd", "cet1_ratio_pct", "leverage_ratio_pct", "roa_pct"]
connection.executemany("INSERT INTO banks VALUES (?, ?, ?, ?, ?, ?, ?, ?)", [[row[column] for column in columns] for row in rows])
for result in connection.execute("SELECT bank_name, report_date, assets_usd, cet1_ratio_pct, leverage_ratio_pct FROM banks ORDER BY cert"):
    print(result)
connection.close()


## Export a comparison and a source recipe

A CSV is easy to share, but cannot carry all evidence by itself. Keep its manifest and the constituent dataset IDs. Do not infer that an institution is available for sale.


In [ ]:
import csv
import hashlib
import json
from pathlib import Path

output = Path("outputs/banksonsale-comparison")
output.mkdir(parents=True, exist_ok=True)
path = output / "comparison.csv"
with path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    writer.writerows([{key: row[key] for key in columns} for row in rows])
recipe = {"publisher": "BanksOnSale", "report_date": "2026-06-30", "certs": [row["cert"] for row in rows], "artifact_sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "panels": [panel.manifest() for panel in panels]}
(output / "comparison.manifest.json").write_text(json.dumps(recipe, indent=2), encoding="utf-8")
assert len(rows) == 3
print(path)


## Continue your own research

Choose peers based on size, charter and business model for the question you are answering. For an existing analysis environment, these same records work with pandas, Polars or DuckDB. For hosted screening and saved research, [explore BanksOnSale](https://www.banksonsale.com/discover). Professional features can require a subscription and display their own data edition.

Original source: FDIC BankFind Suite. Publisher: **BanksOnSale Data**. [Methods and units](https://www.banksonsale.com/methodology).
